# Diffusion-TS

In [ ]:
import torch
import numpy as np
from denoising_diffusion_pytorch.denoising_diffusion_pytorch_1d import Unet1D, GaussianDiffusion1D, Trainer1D, Dataset1D
from tqdm import tqdm
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

### data loading and preprocessing  

In [ ]:
def load_data():
    """
    加载并正确预处理 .npy 数据文件。
    """
    print("开始加载数据...")
    file_path = "./Bamboo500.npy"
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"数据文件未找到: {file_path}。请确保文件路径正确。")

    real_vectors = np.load(file_path)
    data = torch.from_numpy(real_vectors.astype(np.float32))
    data = (data - data.min()) / (data.max() - data.min()) * 2 - 1
    # 原始形状: (500, 64) -> (样本数, 序列长度)
    # 目标形状: (500, 1, 64) -> (样本数 N, 通道数 C, 序列长度 L)
    data = data.unsqueeze(1)
    print(f"数据加载完成。最终数据形状: {data.shape}") 
    return data

### parameters setting

In [ ]:
model = Unet1D(
    dim=64,
    dim_mults=(1, 2, 4, 8),
    channels=1
)

diffusion = GaussianDiffusion1D(
    model,
    seq_length=64,
    timesteps=1000,
    objective='pred_noise'
)

In [ ]:
# 加载数据并创建数据集
training_data = load_data()
dataset = Dataset1D(training_data)

# 配置训练器
trainer = Trainer1D(
    diffusion,
    dataset=dataset,
    train_batch_size=32,
    train_lr=8e-5,
    train_num_steps=5000,         # 训练总步数
    gradient_accumulate_every=2,  
    ema_decay=0.995,              
    amp=True,                     
    save_and_sample_every=200,    # 每200步保存一次模型
    results_folder='./results_id_diffusion'
)

### Trainning

In [ ]:
print("开始训练模型...")
trainer.train()
print("模型训练完成！")

### generate samples

In [9]:
print("开始生成新数据...")

latest_milestone = 10
trainer.load(latest_milestone)

num_samples_to_generate = 6000
batch_size = 100
num_batches = num_samples_to_generate // batch_size

generated_data_list = []

model_to_sample_from = trainer.model
if hasattr(trainer, 'ema'):
    model_to_sample_from = trainer.ema.ema_model
for i in tqdm(range(num_batches), desc="正在生成样本"):
    samples = model_to_sample_from.sample(batch_size=batch_size)
    samples = samples.squeeze(1).cpu().numpy()
    generated_data_list.append(samples)

generated_data_np = np.concatenate(generated_data_list, axis=0)

print(f"数据生成完成。最终数据形状: {generated_data_np.shape}")

output_filename = 'generated_vectors_ddpm_10.npy'
np.save(output_filename, generated_data_np)

print(f"生成的{num_samples_to_generate}个样本已成功保存到 '{output_filename}'")

开始生成新数据...
loading from version 2.2.5


正在生成样本: 100%|██████████| 60/60 [29:32<00:00, 29.54s/it]

数据生成完成。最终数据形状: (6000, 64)
生成的6000个样本已成功保存到 'generated_vectors_ddpm_10.npy'


# Erosion process
erosion process  

In [ ]:
class FractureCurveGenerator:
    def __init__(self, count_fiber, K_Ⅲ, len_x, ce_rate, erosion_epoch, data_file="generated_vectors.npy", save_path='seriesgan_data_6000.npy'):
        self.count_fiber = count_fiber
        self.K_Ⅲ = K_Ⅲ
        self.len_x = len_x
        self.ce_rate = ce_rate
        self.erosion_epoch = erosion_epoch
        self.save_path = save_path
        self.data_file = data_file
        
        # Load pre-generated data
        self.generated_samples = np.load(data_file)
        self.current_index = 0  # Track current data index being used
        
        print(f"Loaded {len(self.generated_samples)} samples from {data_file}")
        print(f"Sample shape: {self.generated_samples.shape}")

    def create_fiber_line(self): 
        '''Description: This function is used to generate the fracture curve'''
        # Get a sample from pre-loaded data
            # If all data is used up, start over with cycling
        if self.current_index >= len(self.generated_samples):
            self.current_index = 0
            print("Warning: Reusing data samples as all samples have been used.")
        
        # Get current sample
        sample = self.generated_samples[self.current_index]
        self.current_index += 1
        
        # Convert to list format
        if isinstance(sample, np.ndarray):
            y_list = sample.tolist()
        else:
            y_list = [sample] if not isinstance(sample, list) else sample
            
        return y_list

    def reset_data_index(self):
        '''Reset the data index to start from the beginning'''
        self.current_index = 0

    def get_random_fiber_line(self):
        '''Get a random sample from the loaded data'''
        random_index = np.random.randint(0, len(self.generated_samples))
        sample = self.generated_samples[random_index]
        
        if isinstance(sample, np.ndarray):
            y_list = sample.tolist()
        else:
            y_list = [sample] if not isinstance(sample, list) else sample
            
        return y_list

    def relu(self, x):
        '''Calculate the ReLU activation function'''
        return np.maximum(0, x)

    def erosion_new(self, list):
        '''Generate a single step of erosion on the curve'''
        new_list = []
        for i in range(len(list)):
            # Determine left and right neighbor values
            if i == 0:
                left_fiber = list[i]
                if len(list) == 1:
                    right_fiber = list[i]
                else:
                    right_fiber = list[i+1]
            elif i == len(list)-1:
                left_fiber = list[i-1]
                right_fiber = list[i]
            else:
                left_fiber = list[i-1]
                right_fiber = list[i+1]
            
            # Calculate erosion for each fiber element
            structural_area = self.relu(list[i]-left_fiber)+self.relu(list[i]-right_fiber)  # Calculate structural area
            erosion_fiber = list[i]-structural_area*self.ce_rate
            new_list.append(erosion_fiber)

        return new_list

    def erosion_with_epoch(self, list, epoch): 
        '''Apply erosion process for multiple epochs'''
        for i in range(epoch):
            list = self.erosion_new(list)
        return list

    def floor_list(self, list, floor):
        '''Adjust the list values to a specified floor level'''
        min_value = min(list) + floor
        adjusted_list = [x - min_value for x in list]
        return adjusted_list

    def revers_list(self, list, floor): 
        '''Reverse the list values and adjust to floor level'''
        inverted_list = [-x for x in list]
        return self.floor_list(inverted_list, floor)

    def get_top_erosion_fiber(self, list, epoch, floor): 
        '''Generate the top erosion fiber through reverse processing'''
        reversed_list = self.revers_list(list, 0)
        erosioned_list = self.erosion_with_epoch(reversed_list, epoch)
        adjusted_list = self.revers_list(erosioned_list, -floor)
        return adjusted_list

    def fibers_resize(self, list, num):
        '''Resize fibers to specified number of points using interpolation'''
        # Create new indices with specified length
        new_indices = np.linspace(0, len(list) - 1, num=num)
        # Use linear interpolation
        resampled_list = np.interp(new_indices, np.arange(len(list)), list)
        resampled_list = [64*x/self.count_fiber for x in resampled_list]
        resampled_list = [round(x, 4) for x in resampled_list]
        return resampled_list

    def get_pair_fibers(self, use_random=False):
        '''Generate a pair of erosion fibers (top and bottom)'''
        if use_random:
            fracture_list = self.get_random_fiber_line()
        else:
            fracture_list = self.create_fiber_line()
            
        erosion_list = self.erosion_with_epoch(fracture_list, self.erosion_epoch)
        erosion_list = self.floor_list(erosion_list, 0)
        top_erosion_list = self.get_top_erosion_fiber(fracture_list, self.erosion_epoch, 0)
        return self.fibers_resize(erosion_list, 64), self.fibers_resize(top_erosion_list, 64)

    def get_fracture_curves(self, data_amount):
        '''Generate processed fracture curve data for training'''
        data_list = []
        array_zero = np.zeros(64)
        
        # Reset index to start from the beginning
        self.reset_data_index()
        
        # Generate basic fracture curve pairs
        for i in range(data_amount):
            a, b = self.get_pair_fibers()
            vector_edge_top = np.array(a)
            vector_edge_bottom = np.array(b)
            list_top_bottom = [array_zero, array_zero, vector_edge_top, vector_edge_bottom]
            data_list.append(list_top_bottom)

        all_data_list = np.array(data_list)
        np.save(self.save_path, all_data_list)
        print(f"Saved {len(all_data_list)} processed samples to '{self.save_path}'")
        return all_data_list

In [17]:
generator = FractureCurveGenerator(
    count_fiber=64,  # based on the dimension of generated_vectors.npy
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_diffusion.npy",
    save_path='generated_vectors_diffusion_0.02_6000.npy'
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_diffusion.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_diffusion_0.02_6000.npy'


In [18]:
generator = FractureCurveGenerator(
    count_fiber=64,  # based on the dimension of generated_vectors.npy
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.01,
    erosion_epoch=500,
    data_file="generated_vectors_diffusion.npy",
    save_path='generated_vectors_diffusion_0.01_6000.npy'
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_diffusion.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_diffusion_0.01_6000.npy'


In [19]:
def perturb_curve(y, noise_level=0.02):
    """
    Simplified curve perturbation function to add realistic roughness
    
    Args:
        y: Original curve data (1D numpy array, float32 format)
        noise_level: Overall noise strength (0.01-0.1 recommended)
    
    Returns:
        Perturbed curve with added roughness (same format as input)
    """
    # Ensure input is numpy array and preserve original dtype
    y = np.asarray(y)
    original_dtype = y.dtype
    
    x = np.linspace(0, 1, len(y))
    
    # 1. Basic Gaussian noise
    basic_noise = np.random.normal(0, noise_level, len(y))
    
    # 2. High-frequency noise for fine-scale variations
    high_freq = np.random.normal(0, noise_level * 0.5, len(y))
    
    # 3. Mid-frequency periodic perturbation
    mid_freq = np.sin(2*np.pi*5*x) * noise_level * 2
    
    # 4. Low-frequency drift for long-term variations
    low_freq = np.cumsum(np.random.normal(0, noise_level/10, len(y)))
    low_freq = low_freq - np.linspace(low_freq[0], low_freq[-1], len(low_freq))
    
    # 5. Nonlinear perturbation: y_new = y + β * sin(γ * y) * noise
    nonlinear_noise = np.random.normal(0, noise_level * 0.8, len(y))
    nonlinear = 0.03 * np.sin(2 * y) * nonlinear_noise
    
    # Combine all perturbations
    y_perturbed = y + basic_noise + high_freq + mid_freq + low_freq + nonlinear
    
    # Preserve original dtype (float32)
    return y_perturbed.astype(original_dtype)

In [20]:
generated_samples = np.load("generated_vectors_diffusion.npy")
perturbed_samples = [perturb_curve(sample, noise_level=0.12) for sample in generated_samples]
np.save("generated_vectors_diffusion_perturbed.npy", perturbed_samples)

In [21]:
generator = FractureCurveGenerator(
    count_fiber=64,  
    K_Ⅲ=1.0,
    len_x=3,
    ce_rate=0.02,
    erosion_epoch=500,
    data_file="generated_vectors_diffusion_perturbed.npy",
    save_path="generated_vectors_diffusion_perturbed_0.02_6000.npy"
)

processed_data = generator.get_fracture_curves(6000)

Loaded 6000 samples from generated_vectors_diffusion_perturbed.npy
Sample shape: (6000, 64)
Saved 6000 processed samples to 'generated_vectors_diffusion_perturbed_0.02_6000.npy'
